# Multi-Robot Path Planning using Hill Climbing

This notebook implements a search-based coordination algorithm for multi-robot systems in a warehouse environment. 

### Key Components:
1. **Hill Climbing Search**: Iteratively optimizes robot paths by minimizing spatiotemporal conflicts.
2. **Conflict Detection**: Identifies vertex collisions (same place, same time) and edge collisions (swapping positions).
3. **Cooperative A***: A high-level heuristic neighbor generator that performs time-expanded pathfinding.
4. **Validation Suite**: Rigorous checks for path feasibility and deadlock scenarios.


In [20]:
import heapq
import time
import random
from collections import defaultdict
from typing import List, Dict, Tuple, Optional, Set
from copy import deepcopy
import sys

# Define missing classes instead of just printing
class Robot:
    """Represents a single robot in the warehouse"""
    def __init__(self, robot_id, grid, start_pos, goal_pos, color='blue'):
        self.robot_id = robot_id
        self.grid = grid
        self.current_pos = start_pos
        self.start_pos = start_pos # Added to match AStarPlanner lookup
        self.goal_pos = goal_pos
        self.path = []
        self.color = color

class Node:
    """Represents a node in the search tree"""
    def __init__(self, position, parent=None, g=0, h=0):
        self.position = position
        self.parent = parent
        self.g = g
        self.h = h
        self.f = g + h

    def __lt__(self, other):
        return self.f < other.f

    def reconstruct_path(self):
        path = []
        current = self
        while current is not None:
            path.append(current.position)
            current = current.parent
        return list(reversed(path))

class ConflictDetector:
    """Detects vertex and swap conflicts"""
    def count_conflicts(self, paths: Dict) -> int:
        return len(self.get_conflict_report(paths))

    def get_conflict_report(self, paths: Dict) -> List[Dict]:
        conflicts = []
        valid_paths = {rid: p for rid, p in paths.items() if p}
        if not valid_paths: return []
        
        max_time = max(len(path) for path in valid_paths.values())
        padded = {rid: p + [p[-1]] * (max_time - len(p)) for rid, p in valid_paths.items()}
        
        rids = list(padded.keys())
        for t in range(max_time):
            pos_map = defaultdict(list)
            for rid in rids:
                pos_map[padded[rid][t]].append(rid)
            for pos, robots in pos_map.items():
                if len(robots) > 1:
                    conflicts.append({'type': 'vertex', 'time': t, 'position': pos, 'robot1': robots[0], 'robot2': robots[1]})
            
            if t < max_time - 1:
                for i, r1 in enumerate(rids):
                    for r2 in rids[i+1:]:
                        if padded[r1][t] == padded[r2][t+1] and padded[r1][t+1] == padded[r2][t] and padded[r1][t] != padded[r1][t+1]:
                            conflicts.append({'type': 'edge', 'time': t, 'robot1': r1, 'robot2': r2, 'edge': (padded[r1][t], padded[r1][t+1])})
        return conflicts

print("Successfully initialized Robot, Node, and ConflictDetector classes.")


Successfully initialized Robot, Node, and ConflictDetector classes.


In [21]:
class GridEnvironment:
    """Warehouse grid representation (from Issam_Laribi)"""
    def __init__(self, filename=None):
        self.grid = []
        self.height = 0
        self.width = 0
        if filename:
            self.load_from_file(filename)
    
    def load_from_file(self, filename):
        self.grid = []
        try:
            with open(filename, 'r') as f:
                for line in f:
                    # Fix: Handle trailing characters properly and ensure consistency
                    row = [char == '.' for char in line.strip() if char in '.T']
                    if row:
                        self.grid.append(row)
            
            # Normalize grid width (ensure all rows are the same length)
            if self.grid:
                max_w = max(len(row) for row in self.grid)
                for i in range(len(self.grid)):
                    if len(self.grid[i]) < max_w:
                        self.grid[i].extend([False] * (max_w - len(self.grid[i])))
                
            self.height = len(self.grid)
            self.width = len(self.grid[0]) if self.height > 0 else 0
            print(f"Ô£ô Grid loaded: {self.width}x{self.height}")
        except Exception as e:
            print(f"Ô£ù Error loading grid: {e}")
    
    def is_valid_position(self, x, y):
        # Correctly check bounds relative to grid dimensions
        return 0 <= y < self.height and 0 <= x < self.width
    
    def is_walkable(self, x, y):
        if not self.is_valid_position(x, y):
            return False
        return self.grid[y][x]
    
    def get_neighbors(self, x, y):
        neighbors = []
        for dx, dy in [(0, 1), (1, 0), (0, -1), (-1, 0)]:
            nx, ny = x + dx, y + dy
            if self.is_walkable(nx, ny):
                neighbors.append((nx, ny))
        return neighbors
    
    def manhattan_distance(self, pos1, pos2):
        return abs(pos1[0] - pos2[0]) + abs(pos1[1] - pos2[1])


In [22]:
class AStarPlanner:
    """A* pathfinding wrapper for independent robot planning (from zakaria)"""
    def __init__(self, grid: GridEnvironment):
        self.grid = grid
    
    def plan_path(self, robot: Robot) -> Optional[List[Tuple[int, int]]]:
        """Plan path for a single robot using A*"""
        start = robot.start_pos
        goal = robot.goal_pos
        
        if not self.grid.is_walkable(*start) or not self.grid.is_walkable(*goal):
            return None
        
        if start == goal:
            return [start]
        
        open_set = []
        start_node = Node(start, g=0, h=self.grid.manhattan_distance(start, goal))
        heapq.heappush(open_set, start_node)
        
        g_score = {start: 0}
        closed_set = set()
        
        while open_set:
            current = heapq.heappop(open_set)
            
            if current.position in closed_set:
                continue
            closed_set.add(current.position)
            
            if current.position == goal:
                return current.reconstruct_path()
            
            neighbors = self.grid.get_neighbors(*current.position)
            for neighbor in neighbors:
                tentative_g = current.g + 1
                
                if neighbor not in g_score or tentative_g < g_score[neighbor]:
                    g_score[neighbor] = tentative_g
                    h = self.grid.manhattan_distance(neighbor, goal)
                    neighbor_node = Node(neighbor, parent=current, g=tentative_g, h=h)
                    heapq.heappush(open_set, neighbor_node)
        
        return None
    
    def plan_all_robots_independent(self, robots: List[Robot]) -> Dict[int, List[Tuple[int, int]]]:
        """Plan paths for all robots independently"""
        paths = {}
        for robot in robots:
            path = self.plan_path(robot)
            paths[robot.robot_id] = path if path else [robot.start_pos]
        return paths


In [ ]:
class HillClimbingSolver:
    """
    Super-Fast Multi-Robot Coordinator.
    Uses pre-built Reservation Tables and optimized lookup to handle 20+ robots in seconds.
    """

    def __init__(self, grid: GridEnvironment, max_iterations: int = 1500, seed: int = None):
        self.grid = grid
        self.max_iterations = max_iterations
        self.conflict_detector = ConflictDetector()
        self.best_global_paths = {}
        self.best_global_conflicts = float('inf')
        self.best_global_len = float('inf')
        if seed is not None: random.seed(seed)

    def _get_reservation_tables(self, paths: Dict, exclude_rid: int):
        # res_t[time] = set of (x,y)
        res_t = defaultdict(set)
        # edges[time] = set of (pos_from, pos_to)
        edges = defaultdict(set)
        
        for rid, path in paths.items():
            if rid == exclude_rid or not path: continue
            for t, pos in enumerate(path):
                res_t[t].add(pos)
            # Edge swap protection
            for t in range(len(path)-1):
                edges[t].add((path[t], path[t+1]))
            # Goal reservation (static)
            gl = path[-1]
            for t in range(len(path), 600):
                res_t[t].add(gl)
        return res_t, edges

    def initialize_paths(self, robots: List[Robot]) -> Dict[int, List[Tuple[int, int]]]:
        shuffled = list(robots)
        random.shuffle(shuffled)
        curr = {}
        for r in shuffled:
            curr[r.robot_id] = self._cooperative_astar_replan(r, [], curr)
        return curr

    def _cooperative_astar_replan(self, robot: Robot, current_path: List[Tuple[int, int]], 
                                  other_paths: Dict, optimize: bool = False) -> List[Tuple[int, int]]:
        res_t, edge_t = self._get_reservation_tables(other_paths, robot.robot_id)
        start, goal = robot.start_pos, robot.goal_pos
        
        # Priority Queue: (f, g, x, y, t)
        open_set = [(self.grid.manhattan_distance(start, goal), 0, start[0], start[1], 0)]
        g_score = {(start[0], start[1], 0): 0}
        parent_map = {(start[0], start[1], 0): None}
        closed_set = set()
        
        max_t = 600
        while open_set:
            f, g, cx, cy, ct = heapq.heappop(open_set)
            
            state = (cx, cy, ct)
            if state in closed_set: continue
            closed_set.add(state)
            
            if (cx, cy) == goal:
                path = []
                while state:
                    path.append((state[0], state[1]))
                    state = parent_map[state]
                return path[::-1]
            
            if ct >= max_t: continue

            # Moves: Orthogonal + Wait (stay)
            for nx, ny in self.grid.get_neighbors(cx, cy) + [(cx, cy)]:
                nt = ct + 1
                if (nx, ny) in res_t[nt]: continue
                if ((nx, ny), (cx, cy)) in edge_t[ct]: continue # Swap collision

                ng = g + 1
                n_state = (nx, ny, nt)
                if n_state not in g_score or ng < g_score[n_state]:
                    g_score[n_state] = ng
                    h = self.grid.manhattan_distance((nx, ny), goal)
                    # Use a very small tie-breaker for deterministic but varied paths
                    nf = ng + h + (random.random() * 0.05 if optimize else 0)
                    parent_map[n_state] = (cx, cy, ct)
                    heapq.heappush(open_set, (nf, ng, nx, ny, nt))

        return current_path if current_path else [start]

    def generate_swap_neighbor(self, robot: Robot, current_path: List[Tuple[int, int]], 
                                other_paths: Dict[int, List[Tuple[int, int]]]) -> Optional[List[Tuple[int, int]]]:
        """
        True hill climbing: shorten path by swapping move orders.
        Takes existing path, tries to find shorter valid path via local modifications.
        """
        if len(current_path) < 4:
            return current_path
        
        best_path = list(current_path)
        best_len = len(current_path)
        
        # 1. Path smoothing: remove redundant intermediate waypoints
        smoothed = self._path_smoothing(current_path, other_paths, robot.robot_id)
        if self._is_valid_path(smoothed, other_paths, robot.robot_id):
            return smoothed
        
        # 2. Try swapping consecutive move segments
        for i in range(1, len(current_path) - 2):
            for j in range(i + 1, len(current_path)):
                # Swap segment [i:j] 
                candidate = current_path[:i] + current_path[i:j][::-1] + current_path[j:]
                if self._is_valid_path(candidate, other_paths, robot.robot_id):
                    if len(candidate) < best_len:
                        best_path = candidate
                        best_len = len(candidate)
        
        return best_path if len(best_path) < len(current_path) else current_path

    def _path_smoothing(self, path: List[Tuple[int, int]], 
                        other_paths: Dict, robot_id: int) -> List[Tuple[int, int]]:
        """Remove unnecessary waypoints that don't change direction"""
        if len(path) < 3:
            return path
        
        smoothed = [path[0]]
        for i in range(1, len(path) - 1):
            # Keep point if it changes direction
            if path[i-1] != path[i+1]:
                smoothed.append(path[i])
        smoothed.append(path[-1])
        
        return smoothed if self._is_valid_path(smoothed, other_paths, robot_id) else path

    def _is_valid_path(self, path: List[Tuple[int, int]], 
                       other_paths: Dict, robot_id: int) -> bool:
        """Check if path has no conflicts with other robots"""
        if not path:
            return False
        
        for t, pos in enumerate(path):
            # Check vertex conflicts
            for oid, opath in other_paths.items():
                if oid != robot_id and t < len(opath):
                    if opath[t] == pos:
                        return False
            
            # Check edge conflicts
            if t < len(path) - 1:
                for oid, opath in other_paths.items():
                    if oid != robot_id and t + 1 < len(opath):
                        if (opath[t] == path[t+1] and opath[t+1] == path[t]):
                            return False
        
        return True

    def hill_climb(self, robots: List[Robot], verbose: bool = True) -> Tuple[Dict, int, List]:
        """Main optimization loop with TRUE swap-based hill climbing"""
        current_paths = self.initialize_paths(robots)
        current_conflicts = self.conflict_detector.count_conflicts(current_paths)
        total_len = sum(len(p) for p in current_paths.values() if p)
        
        if verbose:
            print(f"\nPhase 2: Swap-based Length Optimization... (Start: {current_conflicts} conflicts, {total_len} steps)")
        
        no_imp = 0
        for i in range(self.max_iterations):
            # Randomly pick robot to optimize
            rid = random.choice([r.robot_id for r in robots])
            robot = next(r for r in robots if r.robot_id == rid)
            
            # TRUE hill climbing neighbor generation (swap moves)
            new_p = self.generate_swap_neighbor(robot, current_paths[rid], current_paths)
            
            test_paths = {**current_paths, rid: new_p}
            new_conf = self.conflict_detector.count_conflicts(test_paths)
            new_len = sum(len(p) for p in test_paths.values() if p)
            
            # Accept if: better conflicts OR same conflicts but shorter path
            if new_conf < current_conflicts:
                current_paths = test_paths
                current_conflicts = new_conf
                total_len = new_len
                no_imp = 0
            elif new_conf == current_conflicts and new_len < total_len:
                current_paths = test_paths
                total_len = new_len
                no_imp = 0
            else:
                no_imp += 1
            
            if no_imp > 200:
                break
        
        self.best_global_paths = current_paths
        self.best_global_conflicts = current_conflicts
        return self.best_global_paths, self.best_global_conflicts, []

In [14]:
def validate_paths(paths: Dict[int, List[Tuple[int, int]]], 
                   grid: GridEnvironment, robots: List[Robot]) -> Tuple[bool, List[str]]:
    """Validate that all paths are valid
        
    Args:
        paths (Dict): Robot paths
        grid (GridEnvironment): Grid environment
        robots (List[Robot]): List of robots
            
    Returns:
        Tuple[bool, List[str]]: (is_valid, list_of_errors)
    """
    errors = []

    for robot in robots:
        path = paths.get(robot.robot_id)

        if path is None or len(path) == 0:
            errors.append(f"Robot {robot.robot_id}: No path found")
            continue

        if path[0] != robot.start_pos:
            errors.append(f"Robot {robot.robot_id}: Path does not start at start position")

        if path[-1] != robot.goal_pos:
            errors.append(f"Robot {robot.robot_id}: Path does not end at goal position")

        for i, (x, y) in enumerate(path):
            if not grid.is_walkable(x, y):
                errors.append(f"Robot {robot.robot_id}: Position ({x},{y}) at step {i} is not walkable")

        for i in range(len(path) - 1):
            x1, y1 = path[i]
            x2, y2 = path[i + 1]
            distance = abs(x2 - x1) + abs(y2 - y1)
            if distance > 1:
                errors.append(f"Robot {robot.robot_id}: Invalid move from {path[i]} to {path[i+1]} at step {i}")

    return len(errors) == 0, errors


def print_conflict_details(paths: Dict[int, List[Tuple[int, int]]], 
                          conflict_detector: ConflictDetector, robots: List[Robot]):
    """Print detailed conflict information
    
    Args:
        paths (Dict): Robot paths
        conflict_detector (ConflictDetector): Conflict detector instance
        robots (List[Robot]): List of robots
    """
    conflicts = conflict_detector.get_conflict_report(paths)
    
    if not conflicts:
        print("Ô£ô No conflicts detected!")
        return
    
    print(f"\n{'='*70}")
    print(f"CONFLICT REPORT - Total: {len(conflicts)}")
    print(f"{'='*70}\n")
    
    vertex_conflicts = [c for c in conflicts if c['type'] == 'vertex']
    edge_conflicts = [c for c in conflicts if c['type'] == 'edge']
    
    if vertex_conflicts:
        print(f"Vertex Conflicts ({len(vertex_conflicts)}):")
        for conflict in vertex_conflicts[:10]:
            print(f"  Time {conflict['time']}: Robots {conflict['robot1']} and {conflict['robot2']} "
                  f"both at {conflict['position']}")
        if len(vertex_conflicts) > 10:
            print(f"  ... and {len(vertex_conflicts) - 10} more")
    
    if edge_conflicts:
        print(f"\nEdge Conflicts ({len(edge_conflicts)}):")
        for conflict in edge_conflicts[:10]:
            print(f"  Time {conflict['time']}: Robots {conflict['robot1']} and {conflict['robot2']} "
                  f"traverse {conflict['edge']}")
        if len(edge_conflicts) > 10:
            print(f"  ... and {len(edge_conflicts) - 10} more")


def print_path_summary(paths: Dict[int, List[Tuple[int, int]]], robots: List[Robot]):
    """Print summary of robot paths
    
    Args:
        paths (Dict): Robot paths
        robots (List[Robot]): List of robots
    """
    print(f"\n{'='*70}")
    print(f"PATH SUMMARY")
    print(f"{'='*70}\n")
    
    max_length = max(len(path) for path in paths.values()) if paths else 0
    
    for robot in robots:
        path = paths.get(robot.robot_id, [])
        print(f"Robot {robot.robot_id}:")
        print(f"  Start: {robot.start_pos}, Goal: {robot.goal_pos}")
        print(f"  Path length: {len(path)} steps (max timeline: {max_length} steps)")
        if len(path) <= 10:
            print(f"  Path: {' -> '.join(str(p) for p in path)}")
        else:
            print(f"  Path: {' -> '.join(str(p) for p in path[:5])} -> ... -> {' -> '.join(str(p) for p in path[-2:])}")
        print()


def visualize_grid_with_paths(grid: GridEnvironment, paths: Dict[int, List[Tuple[int, int]]], 
                             robots: List[Robot], timestep: int = 0):
    """Simple ASCII visualization of grid with robot positions
    
    Args:
        grid (GridEnvironment): Grid environment
        paths (Dict): Robot paths
        robots (List[Robot]): List of robots
        timestep (int): Timestep to visualize
    """
    vis_grid = []
    for row in grid.grid:
        vis_grid.append(['.' if cell else 'T' for cell in row])
    
    robot_positions = {}
    for robot in robots:
        path = paths.get(robot.robot_id, [])
        if path:
            pos = path[min(timestep, len(path) - 1)]
            robot_positions[pos] = robot.robot_id
    
    for (x, y), robot_id in robot_positions.items():
        if 0 <= y < len(vis_grid) and 0 <= x < len(vis_grid[0]):
            vis_grid[y][x] = str(robot_id)
    
    print(f"Grid at timestep {timestep}:")
    for row in vis_grid:
        print(''.join(row))
    print()

In [15]:
def detect_deadlocks(paths: Dict[int, List[Tuple[int, int]]], robots: List[Robot]) -> List[Dict]:
    """
    Detect potential deadlocks in robot paths.
    A deadlock is suspected if robots are stationary at a non-goal position 
    for more than a few time steps while others are also stuck nearby.
    """
    deadlocks = []
    max_t = max(len(p) for p in paths.values())
    robot_map = {r.robot_id: r for r in robots}
    
    # Check for "Stationary Deadlock": Robot stops before reaching goal
    for rid, path in paths.items():
        robot = robot_map[rid]
        if path[-1] != robot.goal_pos:
            deadlocks.append({
                'type': 'incomplete_path',
                'robot': rid,
                'last_pos': path[-1],
                'goal': robot.goal_pos,
                'message': f"Robot {rid} stopped at {path[-1]} without reaching goal {robot.goal_pos}"
            })
            
    # Check for "Wait Cycles" or "Eternal Blocking"
    # We look for robots that are in the same spot for the last 5+ steps of the simulation
    # and aren't at their goals.
    stuck_robots = []
    for rid, path in paths.items():
        robot = robot_map[rid]
        # Pad path to max_t to see final state
        final_pos = path[-1]
        if final_pos != robot.goal_pos:
            stuck_robots.append(rid)
            
    if len(stuck_robots) > 1:
        # Check if they are adjacent or blocking each other
        for i in range(len(stuck_robots)):
            for j in range(i + 1, len(stuck_robots)):
                r1_id = stuck_robots[i]
                r2_id = stuck_robots[j]
                p1 = paths[r1_id][-1]
                p2 = paths[r2_id][-1]
                
                # If they are neighbors, it's likely a mutual blockage
                dist = abs(p1[0]-p2[0]) + abs(p1[1]-p2[1])
                if dist <= 1:
                    deadlocks.append({
                        'type': 'mutual_blockage',
                        'robots': [r1_id, r2_id],
                        'positions': [p1, p2],
                        'message': f"Robots {r1_id} and {r2_id} are stuck adjacent to each other at {p1} and {p2}"
                    })
                    
    return deadlocks

def print_deadlock_report(deadlocks: List[Dict]):
    if not deadlocks:
        print("Ô£ô No deadlocks detected in the current paths.")
        return
    
    print(f"\n{'!'*70}")
    print(f"DEADLOCK/STALL REPORT - Found {len(deadlocks)} issues")
    print(f"{'!'*70}\n")
    
    for d in deadlocks:
        print(f"- [{d['type'].upper()}] {d['message']}")


In [ ]:
import time
import random

GRID_FILE_PATH = "grid.txt"

# Better distributed ROBOT_CONFIGS to avoid start/goal bottlenecks
ROBOT_CONFIGS = [
    {"id": 1, "start": (1, 1), "goal": (180, 1)},
    {"id": 2, "start": (5, 2), "goal": (175, 2)},
    {"id": 3, "start": (10, 20), "goal": (170, 20)}, 
    {"id": 4, "start": (15, 21), "goal": (165, 21)},
    {"id": 5, "start": (20, 40), "goal": (160, 40)},
    {"id": 6, "start": (25, 41), "goal": (155, 41)},
    {"id": 7, "start": (5, 50), "goal": (175, 50)},
    {"id": 8, "start": (175, 50), "goal": (5, 50)}, 
    {"id": 9, "start": (50, 5), "goal": (50, 60)},
    {"id": 10, "start": (51, 60), "goal": (51, 5)},
    {"id": 11, "start": (20, 10), "goal": (160, 55)},
    {"id": 12, "start": (160, 55), "goal": (20, 10)},
    {"id": 13, "start": (10, 10), "goal": (10, 50)},
    {"id": 14, "start": (15, 50), "goal": (15, 10)},
    {"id": 15, "start": (30, 5), "goal": (30, 65)},
    {"id": 16, "start": (100, 10), "goal": (100, 60)},
    {"id": 17, "start": (101, 60), "goal": (101, 10)},
    {"id": 18, "start": (150, 30), "goal": (50, 30)},
    {"id": 19, "start": (50, 30), "goal": (150, 30)},
    {"id": 20, "start": (11, 11), "goal": (179, 11)}
]

def run_evaluation_suite():
    print(f"--- High-Performance Coordination Evaluation (20 Robots) ---")
    
    grid = GridEnvironment(GRID_FILE_PATH)
    if not grid.grid: return

    robots = []
    for cfg in ROBOT_CONFIGS:
        def find_legal(p):
            for r in range(10):
                for dx in range(-r, r+1):
                    for dy in range(-r, r+1):
                        nx, ny = p[0]+dx, p[1]+dy
                        if grid.is_walkable(nx, ny): return (nx, ny)
            return p
            
        start, goal = find_legal(cfg["start"]), find_legal(cfg["goal"])
        robots.append(Robot(cfg["id"], grid, start, goal))

    # Higher complexity solver configuration
    solver = HillClimbingSolver(grid, max_iterations=2000, seed=42)
    
    start_t = time.time()
    paths, conflicts, _ = solver.hill_climb(robots, verbose=True)
    elapsed = time.time() - start_t

    is_valid, _ = validate_paths(paths, grid, robots)
    deadlocks = detect_deadlocks(paths, robots)

    return conflicts, deadlocks, is_valid, elapsed, paths, solver

conflicts, deadlocks, is_valid, elapsed, paths, solver = run_evaluation_suite()

print("\n" + "="*40)
print("FINAL INTEGRITY ANALYSIS")
print("="*40)
print(f"Algorithm Success: {'YES' if conflicts == 0 and not deadlocks else 'PARTIAL'}")
print(f"Final Conflicts:   {conflicts}")
print(f"Stalls/Deadlocks:  {len(deadlocks)}")
print(f"System Safety:     {'PASSED' if is_valid else 'FAILED'}")
print(f"Compute Time:      {elapsed:.2f}s")
print("="*40)


--- High-Performance Coordination Evaluation (20 Robots) ---
Ô£ô Grid loaded: 186x71


In [17]:
# Final validation summary
print(f"Final Conflicts: {conflicts}")
print(f"Deadlocks Found: {len(deadlocks)}")
print(f"Success: {conflicts == 0 and len(deadlocks) == 0}")


Final Conflicts: 1
Deadlocks Found: 1
Success: False
